# 05 · Modelo — quem vai sair nos próximos 28 dias?

Treina e compara modelos de classificação para prever o churn em 28 dias, avalia na safra out-of-time e gera a lista dos 50 usuários com maior risco.

**Entrada:** `workspace.tmw_loyalty.abt_churn` · **Saída:** experimentos no MLflow e `output/tables/top50_churn.csv`

> 🚧 Notebook em andamento: a comparação dos algoritmos já está pronta. A avaliação no out of time e o top 50 serão incluídos nas próximas versões.

In [0]:
%run ./00_setup

In [0]:
#%pip install feature_engine xgboost lightgbm catboost -q
#!pip install feature_engine

## 1. Sample: treino, teste e out-of-time

### 1.1 Leitura da ABT
Safras de mar/24 a jul/26 para treino e teste. A safra de ago/26 fica guardada para a avaliação final (out-of-time).

In [0]:
df_train_test = carregar_abt()             # safras de treino (mar/24 a jul/26)
df_oot        = carregar_abt(oot=True)     # safra out-of-time (ago/26), sem sorteio

### 1.2 Uma safra por usuário
Como no professor: uma safra sorteada por usuário, para que ninguém apareça no treino e no teste ao mesmo tempo. A safra out-of-time mantém todas as linhas.

In [0]:

# 1 safra aleatória por usuário, para que o mesmo usuário não apareça no treino e no teste ao mesmo tempo
df_train_test = df_train_test.groupby('IdCliente', group_keys=False).sample(n=1, random_state=42)

print(f'Treino/teste: {len(df_train_test):,} usuários | churn {df_train_test.churn.mean():.1%}'.replace(',', '.'))
print(f'OOT: {len(df_oot):,} usuários | churn {df_oot.churn.mean():.1%}'.replace(',', '.'))

> Com uma safra por usuário, o churn sobe de 47,9% para 72,2%: quem fica aparece em muitas safras, e quem sai aparece em poucas. O sorteio evita que o mesmo usuário esteja no treino e no teste, mas muda a taxa base. Por isso, a seção 5 confere se as probabilidades do modelo batem com o churn real do out-of-time (43,8%).

### 1.3 Target e features


In [0]:
target = 'churn'   # 1 = saiu nos 28 dias seguintes

NAO_FEATURES = ['dtRef', 'IdCliente', target, 'flOOT']   # identificadores, alvo e marcação de OOT
features = df_train_test.columns.drop(NAO_FEATURES).tolist()

CATEGORICAS = df_train_test[features].select_dtypes(exclude='number').columns.tolist()
NUMERICAS = df_train_test[features].select_dtypes(include='number').columns.tolist()

print(f'{len(features)} features, das quais {len(CATEGORICAS)} de texto e {len(NUMERICAS)} numéricas.')
print(f'\nTextuais ({len(CATEGORICAS)}): {CATEGORICAS}')
print(f'\nNuméricas ({len(NUMERICAS)}): {NUMERICAS}')

### 1.4 Separação treino e teste
80/20, estratificado pelo alvo, para manter a mesma taxa de churn nas duas bases.

In [0]:
from sklearn import model_selection

X, y         = df_train_test[features], df_train_test[target]
X_oot, y_oot = df_oot[features], df_oot[target]

# numéricas como decimal: evita o aviso do MLflow e aceita nulos no modelo salvo
X     = X.astype({c: 'float64' for c in NUMERICAS})
X_oot = X_oot.astype({c: 'float64' for c in NUMERICAS})

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    test_size=0.2,
                                                                    stratify=y,        # mantém a proporção de churn entre treino e teste
                                                                    random_state=42)

print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")

O explore precisa ser feito apenas nos dados de treino

## 2. Explore: nulos nas features
Quais features têm nulos e o que o nulo significa em cada uma.

In [0]:
is_na = X_train.isna().mean()
is_na[is_na > 0].sort_values(ascending=False).round(3)

> O nulo quase sempre significa **"o evento não aconteceu"**, e não dado faltando. Por isso, cada tipo de dado recebe um tratamento diferente:

| Tipo de dado | Colunas | % nulo | O que o nulo significa | Tratamento |
| --- | --- | --- | --- | --- |
| **Dias desde o último evento** | `diasUltimoStreak` | 93% | Não fez streak nos últimos 28 dias | Máximo da base |
| | `diasDesdeUltimoEp` | 90% | Nunca concluiu episódio, ou safra antes de mar/2025 | Máximo da base |
| **Plataforma de cursos** | `qtdCursosIniciados`, `qtdCursosFinalizados`, `pctMedioCompleto`, `qtdEpsD28`, `flFezCurso` | 49% | Safra antes de mar/2025, quando a plataforma não existia | 0 + indicador de ausência (`flFezCurso_na`) |
| **Intervalo entre dias ativos** | `avgIntervalo`, `maxIntervalo`, `razaoAtraso` | 36% | Só um dia ativo, sem intervalo para medir | Máximo da base |
| | `stdIntervalo`, `cvIntervalo` | 52% | Menos de três dias ativos, sem variação para medir | Máximo da base |

> As demais features não têm nulos.

## 3. Modify: texto e nulos
Duas etapas do pipeline, ajustadas só no treino: as colunas de texto viram colunas 0/1, e cada grupo de nulos recebe o tratamento acima.

In [0]:
from feature_engine import encoding, imputation

# ---------- Texto -> 0/1 ----------
# uma coluna por categoria; colunas com 2 opções (fezStreak, gastouD28) viram uma coluna só.
# categoria que não apareceu no treino fica com 0 em todas as colunas, sem erro.
texto_0_1 = encoding.OneHotEncoder(variables=CATEGORICAS, drop_last_binary=True)

# ---------- Cadastro: nulo = cliente não encontrado na tabela clientes ----------
CADASTRO   = ['qtdRedes']
cadastro_0 = imputation.ArbitraryNumberImputer(variables=CADASTRO, arbitrary_number=0)

# ---------- Plataforma de cursos: nulo = safra antes de mar/2025 ----------
CURSOS = ['qtdCursosIniciados', 'qtdCursosFinalizados', 'pctMedioCompleto', 'qtdEpsD28', 'flFezCurso']

indicador_cursos = imputation.AddMissingIndicator(variables=['flFezCurso'])        # cria flFezCurso_na
cursos_0         = imputation.ArbitraryNumberImputer(variables=CURSOS, arbitrary_number=0)

# ---------- Dias desde o último evento: nulo = nunca fez ----------
DIAS_DESDE = ['diasUltimoStreak', 'diasDesdeUltimoEp']

# ---------- Intervalo entre dias ativos: nulo = poucos dias ativos para medir ----------
INTERVALOS = ['avgIntervalo', 'maxIntervalo', 'razaoAtraso', 'stdIntervalo', 'cvIntervalo']

maximo = imputation.EndTailImputer(variables=DIAS_DESDE + INTERVALOS, imputation_method='max', fold=1)

# ---------- Segurança: qualquer nulo que sobrar (coluna sem nulo no treino, mas com nulo no teste ou no OOT) ----------
resto_0 = imputation.ArbitraryNumberImputer(arbitrary_number=0)   # sem variables = todas as numéricas

# ---------- Etapas de preparo, na ordem em que o pipeline aplica ----------
etapas_preparo = [
    ('texto_0_1',        texto_0_1),
    ('indicador_cursos', indicador_cursos),   # antes do 0, senão não sobra nulo para marcar
    ('cursos_0',         cursos_0),
    ('maximo',           maximo),
    ('cadastro_0',       cadastro_0),           
]

In [0]:
from sklearn import pipeline

# só para conferir: aplica as etapas de preparo no treino, sem o classificador
preparo = pipeline.Pipeline(steps=etapas_preparo)
X_train_transform = preparo.fit_transform(X_train)

print(f'Colunas: {X_train.shape[1]} antes -> {X_train_transform.shape[1]} depois')
print(f'Nulos restantes: {X_train_transform.isna().sum().sum()}')

In [0]:
#pd.set_option('display.max_rows', None)
#df_train = X_train_transform.copy()          # copy: não altera a base transformada
#df_train[target] = y_train

#sumario = df_train.groupby(target).mean().T  # todas as colunas, inclusive as criadas no preparo
#sumario.columns = ['ficou (0)', 'saiu (1)']
#sumario['razao'] = sumario['saiu (1)'] / sumario['ficou (0)']
#sumario.sort_values('razao').round(2)

### 4.1 Algoritmos
Regressão logística, Random Forest e AdaBoost no mesmo pipeline de preparo, cada um registrado como um run no MLflow.

In [0]:
import os, logging

os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import mlflow
from sklearn import (
    base, ensemble, linear_model, metrics, model_selection,
    neural_network, pipeline, preprocessing, svm, tree,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

USUARIO = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{USUARIO}/churn-teo-me-why")


def com_escala(clf):
    """Logística, MLP e SVM precisam das features na mesma escala."""
    return pipeline.make_pipeline(preprocessing.StandardScaler(), clf)


ALGORITMOS = {
    # ---------- simples e interpretáveis ----------
    "arvore_decisao": tree.DecisionTreeClassifier(
        min_samples_leaf=50,
        random_state=42,
    ),
    "logistica": com_escala(
        linear_model.LogisticRegression(max_iter=2000),
    ),
    # ---------- ensembles de árvores ----------
    "random_forest": ensemble.RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=12,
        random_state=42,
    ),
    "adaboost": ensemble.AdaBoostClassifier(
        n_estimators=1000,
        learning_rate=0.9,
        random_state=42,
    ),
    # ---------- gradient boosting ----------
    "gradient_boosting": ensemble.GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
    ),
    "hist_gradient_boosting": ensemble.HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        random_state=42,
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=15,
        random_state=42,
        verbose=-1,
    ),
    "catboost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=4,
        random_state=42,
        verbose=0,
    ),
    # ---------- outros ----------
    "mlp": com_escala(
        neural_network.MLPClassifier(hidden_layer_sizes=(32, 16), 
                                     max_iter=500, 
                                     early_stopping=True,
                                     random_state=42),
    ),
    "svm": com_escala(
        svm.SVC(probability=True, random_state=42),
    ),
}

mlflow.sklearn.autolog(silent=True)  # liga o registro automático, sem avisos

resultados, modelos = [], {}
for nome, clf in ALGORITMOS.items():
    with mlflow.start_run(run_name=nome):
        # clone: cada algoritmo recebe cópias novas das etapas de preparo
        etapas = [(n, base.clone(etapa)) for n, etapa in etapas_preparo]
        modelo = pipeline.Pipeline(steps=etapas + [("classificador", clf)])

        # validação cruzada no treino (3 partes): é a métrica usada para escolher o modelo
        auc_cv = model_selection.cross_val_score(
            modelo, X_train, y_train, cv=3, scoring="roc_auc"
        ).mean()

        modelo.fit(X_train, y_train)
        auc_train = metrics.roc_auc_score(y_train, modelo.predict_proba(X_train)[:, 1])
        auc_test = metrics.roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
        acc_test = metrics.accuracy_score(y_test, modelo.predict(X_test))

        mlflow.log_metrics({"auc_train": auc_train,
                             "auc_cv": auc_cv, 
                             "auc_test": auc_test})

    modelos[nome] = modelo
    resultados.append(
        {
            "modelo": nome,
            "auc_treino": auc_train,
            "auc_cv": auc_cv,
            "auc_teste": auc_test,
            "acuracia_teste": acc_test,
        }
    )
    print(f"{nome:<24} AUC cv {auc_cv:.3f} | teste {auc_test:.3f}")

mlflow.sklearn.autolog(disable=True)  # desliga: evita runs avulsos nas células seguintes

### 4.2 Referência de comparação.

In [0]:
acc_todos_saem = metrics.accuracy_score(y_test, [1] * len(y_test))   # churn é a classe majoritária (72%)
auc_recencia   = metrics.roc_auc_score(y_test, X_test['recencia'])
print(f'Chutar "todo mundo sai": acurácia {acc_todos_saem:.3f}')
print(f'Recência sozinha: AUC {auc_recencia:.3f}')

### 4.3 Comparação a partir do experimento do MLflow

In [0]:
# 4.3 comparação a partir do experimento do MLflow
runs = mlflow.search_runs(
    experiment_names=[f"/Users/{USUARIO}/churn-teo-me-why"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["start_time DESC"],
)
runs = runs.dropna(subset=["metrics.auc_cv", "metrics.auc_test"])          # só runs com as métricas
runs = runs.drop_duplicates(subset="tags.mlflow.runName", keep="first")    # o mais recente de cada algoritmo

comparacao = (
    runs.rename(columns={
        "tags.mlflow.runName": "modelo",
        "metrics.auc_train":   "auc_treino",
        "metrics.auc_cv":      "auc_cv",
        "metrics.auc_test":    "auc_teste",
    })
    .set_index("modelo")[["auc_treino", "auc_cv", "auc_teste", "run_id"]]
)
comparacao["diferenca"] = comparacao["auc_treino"] - comparacao["auc_teste"]
comparacao.loc["referência: recência", "auc_teste"] = auc_recencia

comparacao.sort_values("auc_cv", ascending=False).round(3)

####  i.Curva ROC dos 4 melhores no teste

In [0]:
top4 = comparacao.drop("referência: recência").sort_values("auc_cv", ascending=False).head(4).index

fig, ax = plt.subplots(figsize=(7, 6))
for nome_modelo, cor in zip(top4, BLUE_PALETTE):
    modelo_run = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[nome_modelo, 'run_id']}/model")
    prob = modelo_run.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f"{nome_modelo} (AUC {metrics.roc_auc_score(y_test, prob):.3f})")

fpr, tpr, _ = metrics.roc_curve(y_test, X_test["recencia"])
ax.plot(fpr, tpr, color=COR_NEUTRA, lw=1.5, ls="--", label=f"recência sozinha (AUC {auc_recencia:.3f})")
ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=":", label="chute (AUC 0,5)")

ax.set(title="Curva ROC no teste: os melhores modelos praticamente empatam",
       xlabel="taxa de falsos positivos (quem ficou, mas o modelo disse que sai)",
       ylabel="taxa de verdadeiros positivos (quem saiu e o modelo acertou)")
ax.legend(loc="lower right")
salvar(fig, "12_roc_comparacao_modelos")

####  ii.Curva ROC do modelo escolhido

In [0]:
MELHOR = comparacao.drop("referência: recência")["auc_cv"].idxmax()
# ou fixe: MELHOR = "random_forest"   # empatado com o catboost; mais simples e sem dependência extra

modelo_final = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[MELHOR, 'run_id']}/model")
print(f"Modelo escolhido: {MELHOR}")

In [0]:
fig, ax = plt.subplots(figsize=(7, 6))
for nome_base, X_base, y_base, cor in [
    ("treino",      X_train, y_train, BLUE_PALETTE[0]),
    ("teste",       X_test,  y_test,  BLUE_PALETTE[1]),
    ("out of time", X_oot,   y_oot,   BLUE_PALETTE[2]),
]:
    prob = modelo_final.predict_proba(X_base)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_base, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f"{nome_base} (AUC {metrics.roc_auc_score(y_base, prob):.3f})")

ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=":", label="chute (AUC 0,5)")
ax.set(title=f"Curva ROC do modelo escolhido ({MELHOR})",
       xlabel="taxa de falsos positivos",
       ylabel="taxa de verdadeiros positivos")
ax.legend(loc="lower right")
salvar(fig, "13_roc_modelo_final")